# FP&A Reporting Tool – V1

Objective: Build a FP&A reporting workflow that cleans financial data, identifies data quality issues, automates Actual vs Budget variance analysis, flags management attention areas, and generates a basic driver-based 3-month forecast.

In [34]:
import pandas as pd
import numpy as np
import os
!pip install xlsxwriter

# Data Loading & Checks

In [35]:
if os.path.exists("sample_financials.xlsx"):
    file_path = "sample_financials.xlsx"
else:
    file_path = "/content/drive/MyDrive/GoogleColab-Data/FPA_V1/sample_financials.xlsx"

df = pd.read_excel(file_path)
print('Raw data loaded:')
display(df.head())

Raw data loaded:


,Month,Business_Unit,Department,Account,Actual,Budget
0,2025-01-31 00:00:00,Core,Sales,Revenue,120000,110000.0
1,2025-01-31 00:00:00,Core,COGS,COGS,52000,48000.0
2,31-01-2025,Core,HR,Salaries,30000,30000.0
3,2025-01-31 00:00:00,Core,Marketing,Marketing,15000,10000.0
4,2025-01-31 00:00:00,Core,Admin,Rent,8000,8000.0


In [36]:
# Provide technical summary of the main dataframe
df.info()

# Check main dataframe for missing values
missing_data = df.isnull().sum()
missing_data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Month          84 non-null     object 
 1   Business_Unit  84 non-null     object 
 2   Department     84 non-null     object 
 3   Account        84 non-null     object 
 4   Actual         83 non-null     object 
 5   Budget         84 non-null     float64
dtypes: float64(1), object(5)
memory usage: 4.6+ KB


,0
Month,11
Business_Unit,11
Department,11
Account,11
Actual,12
Budget,11


In [37]:
# 1. Identify specific error types
# Actual Issues: Missing or non-numeric
actual_issues = df[pd.to_numeric(df['Actual'], errors='coerce').isna() & df['Actual'].notna() | df['Actual'].isna()]

# Budget Issues: Missing or non-numeric
budget_issues = df[pd.to_numeric(df['Budget'], errors='coerce').isna() | df['Budget'].isna()]

# Date Issues: Invalid date strings
date_issues = df[pd.to_datetime(df['Month'], errors='coerce').isna() & df['Month'].notna()]

# 2. Identify rows that are completely empty (Standard cleanup)
null_rows = df[df.isnull().all(axis=1)]

# 3. Create a master issues list for reference
data_issues = pd.concat([actual_issues, budget_issues, date_issues]).drop_duplicates()

print(f'Total rows with issues: {len(data_issues)}')
print(f'- Actual Issues: {len(actual_issues)}')
print(f'- Budget Issues: {len(budget_issues)}')
print(f'- Date Issues: {len(date_issues)}')

display(actual_issues.head())

Total rows with issues: 5
- Actual Issues: 15
- Budget Issues: 11
- Date Issues: 0


,Month,Business_Unit,Department,Account,Actual,Budget
6,2025-01-31 00:00:00,Core,Admin,Other Opex,not available,4000.0
7,NaN,NaN,NaN,NaN,NaN,NaN
15,NaN,NaN,NaN,NaN,NaN,NaN
23,NaN,NaN,NaN,NaN,NaN,NaN
30,2025-02-28 00:00:00,NewBiz,Admin,Other Opex,NaN,3000.0


In [38]:
# 1. Remove rows where all columns are null
df = df.dropna(how='all')

# Identify both NaN values and non-numeric strings ('error', etc.) as missing data
actual_numeric = pd.to_numeric(df['Actual'], errors='coerce')
budget_numeric = pd.to_numeric(df['Budget'], errors='coerce')

missing_financial_data = df[actual_numeric.isna() | budget_numeric.isna()]
print(f"Found {len(missing_financial_data)} rows with missing or erroneous Actual/Budget data.\n")

# 2. Preserve original data and create cleaned columns
df['Actual_Clean'] = actual_numeric.fillna(0)
df['Budget_Clean'] = budget_numeric.fillna(0)

# 3. Standardize the 'Month' column to a consistent date format
df['Month'] = pd.to_datetime(df['Month'], errors='coerce', format='mixed')

failed_dates = df[df['Month'].isna()]

# 4. Quick verification of the fix
print('Data cleansing complete. Current info:\n')
df.info()
display(df[['Actual', 'Actual_Clean', 'Budget', 'Budget_Clean']].head())

Found 4 rows with missing or erroneous Actual/Budget data.

Data cleansing complete. Current info:

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 0 to 94
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Month          84 non-null     datetime64[ns]
 1   Business_Unit  84 non-null     object        
 2   Department     84 non-null     object        
 3   Account        84 non-null     object        
 4   Actual         83 non-null     object        
 5   Budget         84 non-null     float64       
 6   Actual_Clean   84 non-null     float64       
 7   Budget_Clean   84 non-null     float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 5.9+ KB


,Actual,Actual_Clean,Budget,Budget_Clean
0,120000,120000.0,110000.0,110000.0
1,52000,52000.0,48000.0,48000.0
2,30000,30000.0,30000.0,30000.0
3,15000,15000.0,10000.0,10000.0
4,8000,8000.0,8000.0,8000.0




# Variance Analysis

In [39]:
# Calculate raw variance and variance percentage using Clean columns
df["Variance"] = df["Actual_Clean"] - df["Budget_Clean"]
df["Variance_%"] = np.where(
    df["Budget_Clean"] != 0, df["Variance"] / df["Budget_Clean"],0)

# Set favourable / unfavourable
# 1. Define which accounts behave like income
income_accounts = ["Revenue"]
# 2. Classify account type
df["Account_Type"] = np.where(df["Account"].isin(income_accounts), "Income", "Cost")
# 3. Favorable / unfavorable logic
df["Variance_Status"] = np.where(
    ((df["Account_Type"] == "Income") & (df["Variance"] >= 0)) |
    ((df["Account_Type"] == "Cost") & (df["Variance"] <= 0)),
    "Favorable", "Unfavorable")

display(df.head())

,Month,Business_Unit,Department,Account,Actual,Budget,Actual_Clean,Budget_Clean,Variance,Variance_%,Account_Type,Variance_Status
0,2025-01-31,Core,Sales,Revenue,120000,110000.0,120000.0,110000.0,10000.0,0.090909,Income,Favorable
1,2025-01-31,Core,COGS,COGS,52000,48000.0,52000.0,48000.0,4000.0,0.083333,Cost,Unfavorable
2,2025-01-31,Core,HR,Salaries,30000,30000.0,30000.0,30000.0,0.0,0.000000,Cost,Favorable
3,2025-01-31,Core,Marketing,Marketing,15000,10000.0,15000.0,10000.0,5000.0,0.500000,Cost,Unfavorable
4,2025-01-31,Core,Admin,Rent,8000,8000.0,8000.0,8000.0,0.0,0.000000,Cost,Favorable


In [40]:
# 1. Summary by Account using Clean columns
summary_account = df.groupby(['Account', 'Account_Type'])[['Actual_Clean', 'Budget_Clean', 'Variance']].sum().reset_index()
summary_account['Variance_%'] = np.where(
    summary_account['Budget_Clean'] != 0,
    summary_account['Variance'] / summary_account['Budget_Clean'],0)

# Adding Variance Status based on the aggregated totals
summary_account["Variance_Status"] = np.where(
    ((summary_account["Account_Type"] == "Income") & (summary_account["Variance"] >= 0)) |
    ((summary_account["Account_Type"] == "Cost") & (summary_account["Variance"] <= 0)),
    "Favorable", "Unfavorable")

summary_account = summary_account.sort_values('Variance', ascending=False)

# 2. Summary by Business Unit and Account using Clean columns
summary_bu_account = df.groupby(['Business_Unit', 'Account', 'Account_Type'])[['Actual_Clean', 'Budget_Clean', 'Variance']].sum().reset_index()
summary_bu_account['Variance_%'] = np.where(
    summary_bu_account["Budget_Clean"] != 0,
    summary_bu_account['Variance'] / summary_bu_account['Budget_Clean'],0)

# Adding Variance Status based on the aggregated totals
summary_bu_account["Variance_Status"] = np.where(
    ((summary_bu_account["Account_Type"] == "Income") & (summary_bu_account["Variance"] >= 0)) |
    ((summary_bu_account["Account_Type"] == "Cost") & (summary_bu_account["Variance"] <= 0)),
    "Favorable", "Unfavorable")

# Sorting by Business Unit (ascending) and Variance (descending)
summary_bu_account = summary_bu_account.sort_values(['Business_Unit', 'Variance'], ascending=[True, False])

print("--- Total Variances by Account ---")
display(summary_account)

print("\n--- Total Variances by Business Unit and Account ---")
display(summary_bu_account)

--- Total Variances by Account ---


,Account,Account_Type,Actual_Clean,Budget_Clean,Variance,Variance_%,Variance_Status
1,Marketing,Cost,260000.0,165000.0,95000.0,0.575758,Unfavorable
0,COGS,Cost,624000.0,584000.0,40000.0,0.068493,Unfavorable
5,Salaries,Cost,347500.0,338500.0,9000.0,0.026588,Unfavorable
3,Rent,Cost,90000.0,90000.0,0.0,0.000000,Favorable
6,Software,Cost,46900.0,54000.0,-7100.0,-0.131481,Favorable
2,Other Opex,Cost,29000.0,42000.0,-13000.0,-0.309524,Favorable
4,Revenue,Income,1410000.0,1455000.0,-45000.0,-0.030928,Unfavorable



--- Total Variances by Business Unit and Account ---


,Business_Unit,Account,Account_Type,Actual_Clean,Budget_Clean,Variance,Variance_%,Variance_Status
1,Core,Marketing,Cost,120000.0,78000.0,42000.0,0.538462,Unfavorable
4,Core,Revenue,Income,810000.0,780000.0,30000.0,0.038462,Favorable
0,Core,COGS,Cost,342000.0,318000.0,24000.0,0.075472,Unfavorable
5,Core,Salaries,Cost,185000.0,181000.0,4000.0,0.022099,Unfavorable
3,Core,Rent,Cost,48000.0,48000.0,0.0,0.000000,Favorable
2,Core,Other Opex,Cost,22000.0,24000.0,-2000.0,-0.083333,Favorable
6,Core,Software,Cost,24900.0,30000.0,-5100.0,-0.170000,Favorable
8,NewBiz,Marketing,Cost,140000.0,87000.0,53000.0,0.609195,Unfavorable
7,NewBiz,COGS,Cost,282000.0,266000.0,16000.0,0.060150,Unfavorable
12,NewBiz,Salaries,Cost,162500.0,157500.0,5000.0,0.031746,Unfavorable


In [41]:
# Define thresholds and corresponding labels for severity
bins = [-np.inf, -0.30, -0.15, -0.10, 0.10, 0.15, 0.30, np.inf]
labels = ["High", "Medium", "Low", "Normal", "Low", "Medium", "High"]

# Apply severity labels to both summaries using pd.cut
for s in [summary_account, summary_bu_account]:
    s["Severity"] = pd.cut(s["Variance_%"], bins=bins, labels=labels, ordered=False)

# Filter for non-normal issues
issues_account = summary_account[summary_account["Severity"] != "Normal"]
issues_bu = summary_bu_account[summary_bu_account["Severity"] != "Normal"]

print("--- Key Issues by Account (Combined) ---")
display(issues_account.head())

print("\n--- Key Issues by Business Unit ---")
display(issues_bu.head())

--- Key Issues by Account (Combined) ---


,Account,Account_Type,Actual_Clean,Budget_Clean,Variance,Variance_%,Variance_Status,Severity
1,Marketing,Cost,260000.0,165000.0,95000.0,0.575758,Unfavorable,High
6,Software,Cost,46900.0,54000.0,-7100.0,-0.131481,Favorable,Low
2,Other Opex,Cost,29000.0,42000.0,-13000.0,-0.309524,Favorable,High



--- Key Issues by Business Unit ---


,Business_Unit,Account,Account_Type,Actual_Clean,Budget_Clean,Variance,Variance_%,Variance_Status,Severity
1,Core,Marketing,Cost,120000.0,78000.0,42000.0,0.538462,Unfavorable,High
6,Core,Software,Cost,24900.0,30000.0,-5100.0,-0.170000,Favorable,Medium
8,NewBiz,Marketing,Cost,140000.0,87000.0,53000.0,0.609195,Unfavorable,High
9,NewBiz,Other Opex,Cost,7000.0,18000.0,-11000.0,-0.611111,Favorable,High
11,NewBiz,Revenue,Income,600000.0,675000.0,-75000.0,-0.111111,Unfavorable,Low


In [42]:
# 1. Group and sum including Account_Type to preserve it
monthly_summary = df.groupby(['Month', 'Account', 'Account_Type'], observed=False)[['Actual_Clean', 'Budget_Clean']].sum().reset_index()

# 2. Calculate Variance and Status
monthly_summary['Variance'] = monthly_summary['Actual_Clean'] - monthly_summary['Budget_Clean']
monthly_summary['Variance_%'] = np.where(monthly_summary["Budget_Clean"] != 0,
                                         monthly_summary['Variance'] / monthly_summary['Budget_Clean'],0)

# 3. Apply Favorable / Unfavorable logic
monthly_summary['Variance_Status'] = np.where(
    ((monthly_summary['Account_Type'] == 'Income') & (monthly_summary['Variance'] >= 0)) |
    ((monthly_summary['Account_Type'] == 'Cost') & (monthly_summary['Variance'] <= 0)),
    'Favorable', 'Unfavorable')

# 4. Sort for chronological reporting
monthly_summary = monthly_summary.sort_values(['Month', 'Account_Type'])

display(monthly_summary.head())

,Month,Account,Account_Type,Actual_Clean,Budget_Clean,Variance,Variance_%,Variance_Status
0,2025-01-31,COGS,Cost,95000.0,88000.0,7000.0,0.079545,Unfavorable
1,2025-01-31,Marketing,Cost,33000.0,22000.0,11000.0,0.500000,Unfavorable
2,2025-01-31,Other Opex,Cost,2000.0,7000.0,-5000.0,-0.714286,Favorable
3,2025-01-31,Rent,Cost,15000.0,15000.0,0.0,0.000000,Favorable
5,2025-01-31,Salaries,Cost,55000.0,54000.0,1000.0,0.018519,Unfavorable


# 3 Month Forecast

In [43]:
# 1. Prepare base data
forecast_base = df.groupby(['Month', 'Business_Unit', 'Account', 'Account_Type'], observed=False)['Actual_Clean'].sum().reset_index()
last_month = forecast_base['Month'].max()
three_months_ago = last_month - pd.DateOffset(months=2)

# 2. Define Drivers (You can now set these per BU if desired)
growth_rate = 0.02
cogs_rate = 0.40
salary_growth = 0.01

# 3. Build the 3-Month Forecast iterating by Business Unit and Month
future_months = pd.date_range(start=last_month + pd.offsets.MonthEnd(1), periods=3, freq='ME')
business_units = forecast_base['Business_Unit'].unique()
forecast_list = []

for month_idx, month in enumerate(future_months, 1):
    for bu in business_units:
        # Filter historical data for this specific BU
        bu_avg_3m = forecast_base[(forecast_base['Business_Unit'] == bu) & (forecast_base['Month'] >= three_months_ago)]
        bu_last_actual = forecast_base[(forecast_base['Business_Unit'] == bu) & (forecast_base['Month'] == last_month)]
        bu_hist_avg_all = forecast_base[forecast_base['Business_Unit'] == bu].groupby(['Account', 'Account_Type'], observed=False)['Actual_Clean'].mean().reset_index()

        temp = bu_hist_avg_all.copy()
        temp['Business_Unit'] = bu
        temp['Month'] = month

        # Apply Driver: Revenue (BU-specific 3m Avg + Growth)
        rev_mask = temp['Account'] == 'Revenue'
        rev_avg = bu_avg_3m.loc[bu_avg_3m['Account'] == 'Revenue', 'Actual_Clean'].mean() if not bu_avg_3m.loc[bu_avg_3m['Account'] == 'Revenue'].empty else 0
        temp.loc[rev_mask, 'Forecast_Amount'] = round(rev_avg * (1 + growth_rate)**month_idx, 1)

        # Apply Driver: COGS (40% of BU-specific Forecasted Revenue)
        cogs_mask = temp['Account'] == 'COGS'
        current_rev = temp.loc[rev_mask, 'Forecast_Amount'].values[0] if not temp.loc[rev_mask].empty else 0
        temp.loc[cogs_mask, 'Forecast_Amount'] = round(current_rev * cogs_rate, 1)

        # Apply Driver: Salaries (BU-specific Last Actual * growth)
        sal_mask = temp['Account'] == 'Salaries'
        last_sal = bu_last_actual.loc[bu_last_actual['Account'] == 'Salaries', 'Actual_Clean'].sum() if not bu_last_actual.loc[bu_last_actual['Account'] == 'Salaries'].empty else 0
        temp.loc[sal_mask, 'Forecast_Amount'] = round(last_sal * (1 + salary_growth) ** month_idx, 1)

        # Default: Use BU historical average for others
        other_mask = ~temp['Account'].isin(['Revenue', 'Salaries', 'COGS'])
        temp.loc[other_mask, 'Forecast_Amount'] = temp.loc[other_mask, 'Actual_Clean'].round(1)

        forecast_list.append(temp)

forecast_3_month = pd.concat(forecast_list, ignore_index=True)
forecast_3_month = forecast_3_month[['Month', 'Business_Unit', 'Account', 'Account_Type', 'Forecast_Amount']]

display(forecast_3_month.sort_values(['Month', 'Business_Unit']).head(14))

,Month,Business_Unit,Account,Account_Type,Forecast_Amount
0,2025-07-31,Core,COGS,Cost,59160.0
1,2025-07-31,Core,Marketing,Cost,20000.0
2,2025-07-31,Core,Other Opex,Cost,3666.7
3,2025-07-31,Core,Rent,Cost,8000.0
4,2025-07-31,Core,Revenue,Income,147900.0
5,2025-07-31,Core,Salaries,Cost,32320.0
6,2025-07-31,Core,Software,Cost,4150.0
7,2025-07-31,NewBiz,COGS,Cost,44880.0
8,2025-07-31,NewBiz,Marketing,Cost,23333.3
9,2025-07-31,NewBiz,Other Opex,Cost,1166.7


In [44]:
# Pivot the driver-based forecast results
forecast_pivot = forecast_3_month.pivot_table(
    index=['Business_Unit','Account'],
    columns='Month',
    values='Forecast_Amount',
    aggfunc='sum'
).reset_index()

# Format the date columns to Month-Year
forecast_pivot.columns = [col.strftime('%b-%Y') if isinstance(col, pd.Timestamp) else col for col in forecast_pivot.columns]

forecast_methods = pd.DataFrame({
    "Account": ["Revenue", "COGS", "Salaries"],
    "Method": [
        "3 Month Average + Growth Rate",
        "% of Forecast Revenue",
        "Latest Actual + Growth Rate"
    ]
})

display(forecast_methods)
print("\n")
display(forecast_pivot)

,Account,Method
0,Revenue,3 Month Average + Growth Rate
1,COGS,% of Forecast Revenue
2,Salaries,Latest Actual + Growth Rate


,Business_Unit,Account,Jul-2025,Aug-2025,Sep-2025
0,Core,COGS,59160.0,60343.2,61550.1
1,Core,Marketing,20000.0,20000.0,20000.0
2,Core,Other Opex,3666.7,3666.7,3666.7
3,Core,Rent,8000.0,8000.0,8000.0
4,Core,Revenue,147900.0,150858.0,153875.2
5,Core,Salaries,32320.0,32643.2,32969.6
6,Core,Software,4150.0,4150.0,4150.0
7,NewBiz,COGS,44880.0,45777.6,46693.2
8,NewBiz,Marketing,23333.3,23333.3,23333.3
9,NewBiz,Other Opex,1166.7,1166.7,1166.7


# Key Insights Summary

In [45]:
# Calculate basic totals
total_actual = df["Actual_Clean"].sum()
total_budget = df["Budget_Clean"].sum()
total_variance = total_actual - total_budget

# Calculate variance % rounded to 4 decimal places
total_variance_pct = np.where(
    total_budget != 0,
    round(total_variance / total_budget, 4), 0)

# Count favorable/unfavorable statuses
favorable_count = len(df[df["Variance_Status" ] == "Favorable"])
unfavorable_count = len(df[df["Variance_Status"] == "Unfavorable"])

# Count high severity and data quality issues
high_severity_count = len(summary_bu_account[summary_bu_account["Severity"] == "High"])
medium_severity_count = len(summary_bu_account[summary_bu_account["Severity"] == "Medium"])
data_quality_issue_count = len(data_issues)

# Largest favorable and unfavorable variances with account and BU context

largest_fav_row = summary_bu_account[summary_bu_account["Variance_Status"] == "Favorable"].sort_values("Variance", ascending=False).head(1)
largest_unfav_row = summary_bu_account[summary_bu_account["Variance_Status"] == "Unfavorable"].sort_values("Variance", ascending=True).head(1)

largest_fav_desc = (
    f"{largest_fav_row.iloc[0]['Business_Unit']} - "
    f"{largest_fav_row.iloc[0]['Account']}: "
    f"{largest_fav_row.iloc[0]['Variance']:,.1f}"
    if not largest_fav_row.empty else "None")

largest_unfav_desc = (
    f"{largest_unfav_row.iloc[0]['Business_Unit']} - "
    f"{largest_unfav_row.iloc[0]['Account']}: "
    f"{largest_unfav_row.iloc[0]['Variance']:,.1f}"
    if not largest_unfav_row.empty else "None")

# Forecast revenue growth (3M)
# Dividing by the first month's total to get the % growth over the period
forecast_revenue = forecast_3_month[forecast_3_month["Account"] == "Revenue"].groupby("Month")["Forecast_Amount"].sum()
revenue_growth_3M = (forecast_revenue.iloc[-1] - forecast_revenue.iloc[0]) / forecast_revenue.iloc[0]

# Consolidate metrics into a summary table
exec_summary = pd.DataFrame({
    "Metric": [
        "Total Actual", "Total Budget", "Total Variance", "Total Variance %",
        "Favorable Variance Count", "Unfavorable Variance Count", "High Severity Issues",
        "Medium Severity Issues", "Data Quality Issues", "Largest Favorable Variance",
        "Largest Unfavorable Variance", "3-Month Revenue Growth Forecast"
    ],
    "Value": [
        total_actual, total_budget, total_variance, total_variance_pct,
        favorable_count, unfavorable_count, high_severity_count,
        medium_severity_count, data_quality_issue_count, largest_fav_desc,
        largest_unfav_desc, round(revenue_growth_3M, 4)]
})

display(exec_summary)

,Metric,Value
0,Total Actual,2807400.0
1,Total Budget,2728500.0
2,Total Variance,78900.0
3,Total Variance %,0.0289
4,Favorable Variance Count,37
5,Unfavorable Variance Count,47
6,High Severity Issues,3
7,Medium Severity Issues,1
8,Data Quality Issues,5
9,Largest Favorable Variance,"Core - Revenue: 30,000.0"


# Export

In [46]:
# Define the output path
report_output_path = os.path.join(os.path.dirname(file_path), "fpna_report_v1.xlsx")

# Define a dictionary of sheets and the DataFrames they should contain
all_sheets = {
    "Executive_Summary": [(exec_summary, 0)],
    "Summary": [(summary_account, 0), (issues_account, len(summary_account) + 2)],
    "Summary_BU": [(summary_bu_account, 0), (issues_bu, len(summary_bu_account) + 2)],
    "Monthly_Breakdown": [(monthly_summary, 0)],
    "Driver_Forecast": [(forecast_pivot, 0)],
    "Forecast_Methodology": [(forecast_methods, 0)],
    "Three_Month_Forecast": [(forecast_3_month, 0)],
    "Data_Quality_Issues": [(pd.concat([missing_financial_data, failed_dates]).drop_duplicates(), 0)]
}

with pd.ExcelWriter(report_output_path, engine='xlsxwriter') as writer:
    for sheet_name, content in all_sheets.items():
        # Export each DataFrame to its assigned row
        for df_to_write, start_row in content:
            df_to_write.to_excel(writer, sheet_name=sheet_name, index=False, startrow=start_row)

        # Access the worksheet to apply global formatting
        worksheet = writer.sheets[sheet_name]
        worksheet.set_default_row(16)
        worksheet.set_column('A:XFD', 18)

print(f"All reports exported and formatted successfully to: {report_output_path}")

All reports exported and formatted successfully to: /content/drive/MyDrive/GoogleColab-Data/FPA_V1/fpna_report_v1.xlsx
